# 07 — Net Realization & Best Market Recommendation
Task 8: move from “which market has the highest predicted price” to “which market is actually most profitable after transport cost, buyer reliability, and perishability risk.”

## 1. Load Task 7 results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

market_results = pd.read_csv(
    "../data/processed/market_predictions.csv"
)

market_performance = pd.read_csv(
    "../data/processed/market_performance.csv"
)

print(market_results.shape)
market_results.head()

## 2. Understand the current data

In [ ]:
print(market_results.columns.tolist())

## 3. Define farmer inputs
For the prototype these are hard-coded; later they'll come from the application UI.

In [ ]:
farmer_quantity = 1000       # kg
vehicle_capacity = 1000      # kg

## 4. Market distance
⚠️ Prototype data only — not real routing distances. Task 9 should replace this with an actual routing/distance calculation.

In [ ]:
market_distance = pd.DataFrame({
    "market": market_performance["market"],
    "distance_km": np.random.randint(
        10,
        150,
        size=len(market_performance)
    )
})

market_distance.head()

## 5. Transport cost
Configurable transport-cost model.

In [ ]:
cost_per_km = 12

market_distance["transport_cost"] = (
    market_distance["distance_km"]
    * cost_per_km
)

market_distance["transport_cost_per_kg"] = (
    market_distance["transport_cost"]
    / farmer_quantity
)

## 6. Merge price + distance

In [ ]:
recommendation_df = market_performance.merge(
    market_distance,
    on="market",
    how="left"
)

recommendation_df.head()

## 7. Calculate expected revenue
Predicted price is per quintal, so convert the farmer's quantity accordingly.

In [ ]:
quantity_quintal = farmer_quantity / 100

recommendation_df["predicted_revenue"] = (
    recommendation_df["predicted_avg"]
    * quantity_quintal
)

## 8. Buyer reliability
⚠️ Prototype/synthetic data only — not real buyer data. Later this should come from actual platform interactions (completed transactions, payment delays, cancellation rate, dispute rate, fulfillment rate).

In [ ]:
buyer_reliability = pd.DataFrame({
    "market": market_performance["market"],
    "buyer_reliability_score": np.random.uniform(
        0.70,
        0.98,
        size=len(market_performance)
    )
})

## 9. Merge reliability

In [ ]:
recommendation_df = recommendation_df.merge(
    buyer_reliability,
    on="market",
    how="left"
)

## 10. Create reliability penalty
High reliability → small penalty, low reliability → larger penalty.

In [ ]:
risk_weight = 0.10

recommendation_df["reliability_penalty"] = (
    (
        1
        - recommendation_df[
            "buyer_reliability_score"
        ]
    )
    * risk_weight
    * recommendation_df["predicted_revenue"]
)

## 11. Add perishability risk
Perishable crops lose value the longer transport/storage takes. This rate is a prototype assumption, not a validated tomato spoilage curve.

In [ ]:
perishability_rate_per_day = 0.03

average_speed_kmph = 40

recommendation_df["travel_time_hours"] = (
    recommendation_df["distance_km"]
    / average_speed_kmph
)

recommendation_df["travel_time_days"] = (
    recommendation_df["travel_time_hours"]
    / 24
)

## 12. Calculate perishability penalty

In [ ]:
recommendation_df["perishability_penalty"] = (
    recommendation_df["predicted_revenue"]
    * perishability_rate_per_day
    * recommendation_df["travel_time_days"]
)

## 13. Calculate net realization
The core equation: predicted revenue minus transport cost minus the reliability and perishability penalties.

In [ ]:
recommendation_df["net_realization"] = (
    recommendation_df["predicted_revenue"]
    - recommendation_df["transport_cost"]
    - recommendation_df["reliability_penalty"]
    - recommendation_df["perishability_penalty"]
)

## 14. Calculate net realization per quintal
Puts every market on a fair, comparable footing.

In [ ]:
recommendation_df["net_per_quintal"] = (
    recommendation_df["net_realization"]
    / quantity_quintal
)

## 15. Rank the markets

In [ ]:
recommendation_df = (
    recommendation_df
    .sort_values(
        "net_realization",
        ascending=False
    )
    .reset_index(drop=True)
)

recommendation_df["rank"] = (
    recommendation_df.index + 1
)

recommendation_df[
    [
        "rank",
        "market",
        "predicted_avg",
        "distance_km",
        "transport_cost",
        "buyer_reliability_score",
        "net_realization",
        "net_per_quintal"
    ]
].head(10)

## 16. Identify the recommended market

In [ ]:
best_market = recommendation_df.iloc[0]

print(
    "Recommended Market:",
    best_market["market"]
)

print(
    "Expected Net Realization:",
    best_market["net_realization"]
)

print(
    "Expected Net / Quintal:",
    best_market["net_per_quintal"]
)

## 17. Create the recommendation graph
Communicates: highest predicted price ≠ always the best market — net realization is what matters.

In [ ]:
top_markets = recommendation_df.head(10)

plt.figure(figsize=(10, 5))

plt.bar(
    top_markets["market"],
    top_markets["net_per_quintal"]
)

plt.xlabel("Market")
plt.ylabel("Expected Net Realization / Quintal")
plt.title(
    "Best Markets by Expected Net Realization"
)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 18. Compare price vs net realization
Demonstrates why the system considers more than just price.

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    recommendation_df["predicted_avg"],
    recommendation_df["net_per_quintal"]
)

plt.xlabel("Predicted Market Price")
plt.ylabel("Expected Net Realization / Quintal")
plt.title(
    "Predicted Price vs Net Realization"
)

plt.tight_layout()
plt.show()

## 19. Save the recommendation

In [ ]:
recommendation_df.to_csv(
    "../data/processed/final_market_recommendations.csv",
    index=False
)

print(
    "Final market recommendations saved."
)

## 20. Create a reusable recommendation function
This is what the backend will eventually call.

In [ ]:
def recommend_best_market(
    prediction_df,
    quantity_kg,
    cost_per_km=12,
    average_speed_kmph=40,
    risk_weight=0.10,
    perishability_rate_per_day=0.03
):

    quantity_quintal = quantity_kg / 100

    result = prediction_df.copy()

    result["transport_cost"] = (
        result["distance_km"]
        * cost_per_km
    )

    result["predicted_revenue"] = (
        result["predicted_avg"]
        * quantity_quintal
    )

    result["travel_time_hours"] = (
        result["distance_km"]
        / average_speed_kmph
    )

    result["travel_time_days"] = (
        result["travel_time_hours"]
        / 24
    )

    result["reliability_penalty"] = (
        (1 - result["buyer_reliability_score"])
        * risk_weight
        * result["predicted_revenue"]
    )

    result["perishability_penalty"] = (
        result["predicted_revenue"]
        * perishability_rate_per_day
        * result["travel_time_days"]
    )

    result["net_realization"] = (
        result["predicted_revenue"]
        - result["transport_cost"]
        - result["reliability_penalty"]
        - result["perishability_penalty"]
    )

    result["net_per_quintal"] = (
        result["net_realization"]
        / quantity_quintal
    )

    result = result.sort_values(
        "net_realization",
        ascending=False
    )

    result["rank"] = range(
        1,
        len(result) + 1
    )

    return result

## 21. Test the function

In [ ]:
final_recommendations = recommend_best_market(
    recommendation_df,
    quantity_kg=1000
)

final_recommendations[
    [
        "rank",
        "market",
        "predicted_avg",
        "net_realization",
        "net_per_quintal"
    ]
].head()

## ⚠️ Very important for your SIH project

At this stage, prototype assumptions have been introduced for **distance**, **transport cost**, **buyer reliability**, and **perishability**.

Don't present these as real data — don't claim, for example, that buyer reliability comes from government sources. Since Kadak Bhav is specifically designed to avoid depending on a single government data source, the final architecture should draw buyer reliability from actual farmer/buyer platform interactions, and distance/logistics from independently available routing sources, replacing today's `np.random` placeholders.

## Next step
Task 9 replaces the prototype distance table with a real routing/distance calculation, and starts sourcing buyer reliability from actual platform transaction history instead of `np.random.uniform`.